# FlowGT Radar
## 从零搭一个「捡漏器」，并且看得见每一分钱

---

**这份笔记本的用法**：从上往下一格一格跑。每个代码格前面都有一个灰框，
写清楚 **干什么 / 输入 / 花费 / 应该看到什么 / 看到别的说明什么**。

**它同时是一份教材。** 跑完之后你应该能站在白板前，把下面这几件事讲给别人听：

| 概念 | 一句话 |
|---|---|
| **harness** | 包在模型调用外面的那圈代码 |
| **context window** | 模型一次能看见的全部文字 |
| **context engineering** | 决定「哪些字进去、按什么顺序」 |
| **prompt caching** | 让不变的那段只付一次全价 |
| **MCP**（Model Context Protocol） | 让模型连上外部工具的开放标准 |
| **Agent Skill** | 一个文件夹，按需加载的专业知识 |
| **workflow vs agent** | 流程图画得出来的是 workflow，画不出来的才是 agent |

> 讲不出来就是没学会。这是费曼学习法的判据，也是这份笔记本的验收标准。

---
# Part 0 · 先看地图

## 0.1 我们要造的东西

```
                     ┌──────────────────────────────┐
                     │  痛点 THE PROBLEM            │
                     │  招聘方写标题时              │
                     │  不考虑你搜不搜得到          │
                     └───────────────┬──────────────┘
                                     │
          ┌──────────────────────────┴──────────────────────────┐
          │                                                     │
   标题写 AI 的岗位                                    标题写 Backend Developer
   200 人投                                            正文第三页才提 RAG
   竞争惨烈                                            5 人投 · 活是一样的
          │                                                     │
          └──────────────────────────┬──────────────────────────┘
                                     │
                          ┌──────────▼──────────┐
                          │   RADAR             │
                          │   把正文读一遍       │
                          └──────────┬──────────┘
                                     │
                   ┌─────────────────┴─────────────────┐
                   │                                   │
        ┌──────────▼──────────┐            ┌───────────▼──────────┐
        │ AI 渗透度  0-5      │            │ 新人友好度  0-5      │
        │ ai_depth            │     ×      │ newcomer_fit         │
        │ 这活有多少 AI       │            │ 新人投了有没有戏     │
        └──────────┬──────────┘            └───────────┬──────────┘
                   └─────────────────┬─────────────────┘
                                     │
                       ┌─────────────▼─────────────┐
                       │  机会指数 opportunity      │
                       │  乘法，不是加法            │
                       └─────────────┬─────────────┘
                                     │
                   ┌─────────────────┴─────────────────┐
          ┌────────▼────────┐               ┌──────────▼─────────┐
          │ 朝会员读        │               │ 朝雇主读           │
          │ 友好度高=值得投 │               │ 友好度低=你招不到人 │
          └─────────────────┘               └────────────────────┘
```

## 0.2 为什么是**乘法**

任何一个是 0，这岗位对会员就毫无意义。**加法会让「AI 满分但要 10 年经验」看起来还不错。**

```
渗透度 5 × 友好度 1 =  5    团队做 AI 很久了 → 门槛已经立起来了
渗透度 3 × 友好度 5 = 15    团队刚开始碰 AI → 门槛还没立 ← 甜蜜区 sweet spot
```

**反直觉但正确：3 分比 5 分值钱。**

## 0.3 文件夹架构 —— 以及为什么这么摆

```
flowgt-radar/
├── config.md              ← 密钥 API key。被 .gitignore 挡住，永不入库
├── .gitignore             ← 【先于第一次 commit 存在】
├── radar.ipynb            ← 你正在读的这个（教学 + 实验台）
├── build_notebook.py      ← 生成 ipynb 的脚本
│
├── skills/                ← 【知识】放这里，不放代码里
│   └── ai-job-scoring/
│       ├── SKILL.md       ← 第一层：这个 skill 是什么、什么时候用
│       └── references/
│           └── rubric.md  ← 第二层：详细评分表，需要时才读
│
├── harness/               ← 【机械】放这里，和知识分开
│   ├── client.py          ← 怎么调模型
│   ├── ledger.py          ← 记账
│   └── budget.py          ← 预算闸
│
└── out/                   ← 产物。.gitignore 挡住，随时可重建
```

### 三条设计理由 · why this shape

**① 知识和机械分开。** `skills/` 里是「怎么判断一个岗位」，`harness/` 里是
「怎么调 API、怎么记账」。改判断标准不该碰调用代码，反过来也一样。
这也是为什么 rubric 是一个 **`.md` 文件**而不是 Python 字符串 —— 
**`.md` 是给人改的**，而改它的人（你）不写 Python。

**② `skills/` 的两层结构不是随便分的。** 它照着 Anthropic 的 Agent Skills 规范摆：
`SKILL.md` 是「目录」，`references/` 是「附录」。官方管这个叫 **progressive disclosure**
（渐进披露）—— 模型先读元信息，需要时才读细节，**不把所有东西一次塞进 context window**。

**③ 密钥的位置是一条硬规矩。** `.gitignore` 必须**先于**第一次 commit 存在。
反过来的话密钥已经进了 git 历史，而从历史里删一个密钥要重写历史。

> 📖 Agent Skills 规范：<https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills>
> 官方定义：skill 是 *"organized folders of instructions, scripts, and resources
> that agents can discover and load dynamically"*。

### 🔨 刻意练习 · deliberate practice

1. 用你自己的话，把 0.1 那张图画在纸上，不看笔记本。画不出来就回去再读一遍。
2. 回答：如果改成加法，哪一类岗位会被错误地排到前面？举一个具体例子。
3. 看 0.3 的目录树，回答：为什么 rubric 是 `.md` 不是 `.py`？

---
# Part 1 · 五个必须搞懂的词

这一部分不跑代码，但**它决定你后面每一格能不能看懂**。

## 1.1 context window（上下文窗口）

模型**一次能看见的全部文字**。你的问题、你给的资料、它自己之前说的话，全在里面。

```
┌─────────── context window ───────────┐
│ system: 你是新西兰就业市场分析员…    │  ← 不变的部分
│ user:   职位 Backend Developer …     │  ← 每次都变的部分
│ ─────────────────────────────────    │
│ assistant: {"ai_depth": 3, …}        │  ← 模型的回答也占位置
└──────────────────────────────────────┘
```

**窗口是有限的，而且每个字都要付钱。** 所以「往里放什么、按什么顺序放」是一门手艺 ——
这门手艺叫 **context engineering**。

## 1.2 harness（外壳 / 支架）

**包在模型调用外面的那圈代码。** 不是官方术语，是工程圈的俗称。

```
      你的代码                模型
   ┌──────────────┐
   │   HARNESS    │
   │ ┌──────────┐ │
   │ │ 组装提示词│ │
   │ │ 调用      │─┼────────→  DeepSeek
   │ │ 记账      │ │  ←────────  回答 + usage
   │ │ 预算闸    │ │
   │ │ 解析 JSON │ │
   │ │ 出错重试  │ │
   │ └──────────┘ │
   └──────────────┘
```

**没有 harness 也能调模型** —— 一行 `client.chat.completions.create()` 就够。
但那样你不知道花了多少钱、不知道缓存有没有命中、超支了没人拦、
模型吐了半句 JSON 你的程序就崩了。

> **harness 是「让一次能跑」和「让一千次能跑」之间的全部差别。**

## 1.3 MCP（Model Context Protocol）

一个**开放标准**，让模型连上外部的工具和数据。官网 <https://modelcontextprotocol.io>

```
   Claude / 任何 AI 应用              MCP Server
   ┌──────────────┐                ┌──────────────┐
   │  MCP Client  │◄──── 标准协议 ──►│  你的数据库   │
   └──────────────┘                │  你的 API     │
                                   │  你的文件      │
                                   └──────────────┘
```

**类比：MCP 之于 AI 工具，等于 USB-C 之于充电器。** 以前每个应用要为每个数据源
单独写一套对接；有了标准，写一次 server，所有支持 MCP 的客户端都能用。

**这个笔记本里没有 MCP，而且是故意的。** 我们只是「调一次模型、拿回一个 JSON」，
模型不需要自己去查任何东西。**用不上的标准不要引入** —— 这是后面 1.5 的伏笔。

## 1.4 Agent Skill（技能）

一个**文件夹**，里面装着某项专门知识，模型按需加载。

官方定义：*"organized folders of instructions, scripts, and resources that agents
can discover and load dynamically to perform better at specific tasks"*

```
skills/ai-job-scoring/
├── SKILL.md          ← 第一层：我是谁、什么时候该用我（几百字）
└── references/
    └── rubric.md     ← 第二层：完整评分表（几千字，需要时才读）
```

**核心机制叫 progressive disclosure（渐进披露）**：官方的比喻是
*"先看目录，再看章节，最后才看附录"* —— **不把所有东西一次塞进 context window**，
因为窗口有限而且每个字都要付钱。

**skill 和 MCP 的关系：互补，不是二选一。**
MCP 解决「模型怎么拿到外部东西」，skill 解决「模型该怎么做这件事」。

**在这个项目里，`rubric.md` 就是一个 skill。** 它是「怎么给岗位打分」这门知识，
和调用代码分开放 —— 所以你可以改它而不碰一行 Python。

## 1.5 workflow vs agent —— 这个笔记本最重要的判断

官方定义（Anthropic《Building Effective Agents》）：

| | 定义 |
|---|---|
| **Workflow** | *"LLMs and tools are orchestrated through **predefined code paths**"*（走预先写好的路径） |
| **Agent** | *"LLMs **dynamically direct their own processes** and tool usage"*（模型自己决定下一步） |

**判据（我给你的口诀）：**

> **流程图画得出来的，是 workflow。画不出来的，才是 agent。**

Radar 的流程图：

```
  每个岗位 ─→ 读正文 ─→ 打两个分 ─→ 相乘 ─→ 排序 ─→ 完
```

**画得出来 → Radar 是 workflow，不是 agent。**

而 1v1 录音蒸馏 rubric 的流程：

```
  听到一句判断 ─→ 查 rubric 有没有 ─→ 没有 ─→ 回头找上下文
                                    │              ↓
                                    │        查这人历史说过没有
                                    │              ↓
                                    └────── 说过？那是加强证据，不是新规则
                                                   ↓
                                             起草一条新规则
```

**每一步都取决于上一步查到了什么，步数事先不知道 → 这才是 agent。**

### 为什么这个区分值钱

官方原话：*"Start with simple prompts... and add multi-step agentic systems
**only when simpler solutions fall short**"*，加复杂度要 *"only when it
demonstrably improves outcomes"*。

把 workflow 做成 agent 的代价：**更贵、更慢、每次结果不一样**。
而排序这种事，最需要的恰恰是**今天和明天算出来一样**。

> 📖 <https://www.anthropic.com/engineering/building-effective-agents>

### 🔨 刻意练习 · deliberate practice

1. 不看笔记本，用一句话解释 harness。再说出三件「没有 harness 就做不到」的事。
2. MCP 和 Skill 的区别是什么？用「USB-C」和「新员工入职手册」两个比喻各说一遍。
3. 画出 Radar 的流程图。然后回答：它是 workflow 还是 agent？为什么？
4. 想一个你自己工作里的任务，判断它是 workflow 还是 agent。说出理由。

---
# Part 2 · 环境、钥匙、模型

> **这一格干什么** 装两个 Python 包。`openai` 只是当 HTTP 客户端用 —— DeepSeek 的接口和它兼容，**和 OpenAI 这家公司无关**。  
> **输入 input** 无  
> **花费 cost** 不花钱  
> **你应该看到 expected** 一行 `装完了 / installed`
  
> **看到别的说明** 报网络错就重跑一次

In [ ]:
# openai     → HTTP client（DeepSeek is API-compatible with it）
# playwright → 真浏览器，用来抓 JD 正文 / a real browser, to fetch job-ad bodies
%pip install -q openai playwright
print('装完了 / installed')

> **这一格干什么** 下载 Chromium 内核。  
> **输入 input** 无  
> **花费 cost** 不花钱  
> **你应该看到 expected** 下载进度，或 `is already installed`。第一次一两分钟
  
> **看到别的说明** 卡住就去终端跑 `python -m playwright install chromium`

In [ ]:
!python -m playwright install chromium 2>&1 | tail -2

## 2.1 钥匙 —— 一条不能破的规矩

**API key 从文件读，绝不写进代码。**

写进代码 = 写进 git = 泄漏。而且从 git 历史里删一个密钥**要重写历史** ——
那是能避免就必须避免的事。所以：

```
config.md           ← 密钥住这里
.gitignore          ← 写着 config.md，且【先于第一次 commit 存在】
```

> **这一格干什么** 读密钥、查余额。  
> **输入 input** `config.md`  
> **花费 cost** 不花钱  
> **你应该看到 expected** 钥匙前 7 位 + 长度，然后 `余额 CNY 10.00`
  
> **看到别的说明** `AssertionError` = 文件不对；`HTTPError 401` = 密钥无效

In [ ]:
import pathlib, json, urllib.request, time

KEY = pathlib.Path('config.md').read_text(encoding='utf-8').strip()
assert KEY.startswith('sk-'), 'config.md 第一行应该是 sk- 开头的密钥'
# 只打印前 7 位：够你确认读对了文件，又不会把密钥留在 notebook 输出里。
# Enough to confirm the right file was read, without leaving the key in the
# notebook's saved output — which does get committed.
print(f'钥匙 {KEY[:7]}…（{len(KEY)} 位）')

def api(path):
    """DeepSeek 管理接口（余额、模型列表）。免费。/ Free management endpoints."""
    req = urllib.request.Request(f'https://api.deepseek.com{path}',
                                 headers={'Authorization': f'Bearer {KEY}'})
    return json.load(urllib.request.urlopen(req, timeout=20))

bal = api('/user/balance')
cny = next(b for b in bal['balance_infos'] if b['currency'] == 'CNY')
START_BALANCE = float(cny['total_balance'])
print(f"余额 CNY {START_BALANCE:.2f}　可用 {bal['is_available']}")

## 2.2 有哪些模型？**问 API，别问人**

写这份笔记本时，我凭记忆以为模型叫 `deepseek-chat` 和 `deepseek-reasoner`。
**两个都是错的。**

> **规矩：凡是会变的东西（模型名、价格、余额），一律现场查，不写死在脑子里。**

> **这一格干什么** 列出这个 key 能用的模型。  
> **输入 input** 无  
> **花费 cost** 不花钱  
> **你应该看到 expected** `deepseek-v4-flash` 和 `deepseek-v4-pro`
  
> **看到别的说明** 名字对不上就把下面两个常量改成打印出来的

In [ ]:
models = [m['id'] for m in api('/models')['data']]
print('可用模型 / models available:')
for m in models: print('  ', m)

FLASH = 'deepseek-v4-flash'   # 便宜 · 粗筛 triage
PRO   = 'deepseek-v4-pro'     # 贵   · 细读 deep read
assert FLASH in models and PRO in models, f'名字对不上 / mismatch: {models}'
print('\n✓ 对得上')

---
# Part 3 · 钱：价目表、缓存、台账

## 3.1 价目表

单位 **USD / 每 1M tokens**。2026-08-11 抄自官方文档。

| 模型 | 输入·命中缓存 cache hit | 输入·没命中 cache miss | 输出 output |
|---|---|---|---|
| `deepseek-v4-flash` | $0.0028 | $0.14 | $0.28 |
| `deepseek-v4-pro` | $0.003625 | $0.435 | $0.87 |

⚠️ 官方同一页写着「近期将大幅上调价格」。**这张表要定期核。**

## 3.2 👉 整个笔记本最值钱的一句话

```
flash   0.14    ÷ 0.0028   ≈  50 倍
pro     0.435   ÷ 0.003625 ≈ 120 倍
```

> **命中缓存比没命中便宜 50 到 120 倍。差两个数量级。**

### prompt caching 是什么

模型服务商会把**你提示词开头那一段**存起来。下次你发来的提示词如果**开头完全一样**，
它就直接复用，只收极低的费用。

```
第一次   [═══ rubric 2400 tok ═══][ 岗位A 500 tok ]
                    ↓ 存起来
第二次   [═══ rubric 2400 tok ═══][ 岗位B 500 tok ]
          ↑ 命中！只付 1/50        ↑ 全价
```

**所以规矩是：不变的放最前面（system），每次变的放最后（user）。**
顺序反过来，一次都命中不了。

### ⚠️ 但有前提：提示词得够长

我实测过（数字，不是说法）：

```
提示词  100 tok：第 1 次 miss 100    第 2 次 miss 100        ← 太短，根本不进缓存
提示词 2400 tok：第 1 次 miss 2420   第 2 次 hit 2304 (95.3%)
```

> **「rubric 写详细一点」不是啰嗦，是省钱。**

> **这一格干什么** 存价目表，打印倍数。  
> **输入 input** 无  
> **花费 cost** 不花钱  
> **你应该看到 expected** 两行：50 倍 和 120 倍

In [ ]:
# USD / 1M tokens。改价【只改这里】/ the only place prices live.
PRICES = {
    'deepseek-v4-flash': {'hit': 0.0028,   'miss': 0.14,  'out': 0.28},
    'deepseek-v4-pro':   {'hit': 0.003625, 'miss': 0.435, 'out': 0.87},
}
USD_TO_CNY = 7.1   # 粗略汇率，只为了和 10 元余额对得上

for m, p in PRICES.items():
    print(f"{m:<20} cache miss 比 hit 贵 {p['miss']/p['hit']:>5.0f} 倍")

## 3.3 台账 + 预算闸 —— 这就是 harness

下面这一格**就是 Part 1.2 讲的 harness**。它包住每一次调用，回答四个问题：

1. 送进去多少 token？其中多少 **cache hit**？
2. 吐出来多少？其中多少是**思考 reasoning tokens**？
3. 这次花了多少？
4. 累计多少、还剩多少？

**外加一道硬闸：超预算直接抛异常。**
不是打印警告 —— 警告会被划过去。是抛异常，让后面的格子跑不下去。

> 这就是「让一次能跑」和「让一千次能跑」之间的差别。

> **这一格干什么** 定义 `ask()` 和 `estimate()`。  
> **输入 input** 无  
> **花费 cost** **不花钱** —— 这格只定义函数，不调模型  
> **你应该看到 expected** `harness 就绪　预算上限 CNY 2.00`

In [ ]:
from openai import OpenAI
client = OpenAI(api_key=KEY, base_url='https://api.deepseek.com')

# ══ 限制 limits ══════════════════════════════════════════════════════════
BUDGET_CNY = 2.00   # 这个笔记本最多花这么多 / hard ceiling for this notebook
MAX_JOBS   = 12     # 一批最多评几个 / cap per batch

LEDGER = []
def spent_cny(): return sum(r['cost_usd'] for r in LEDGER) * USD_TO_CNY

class BudgetExceeded(RuntimeError):
    """刻意做成异常而不是警告 —— 警告会被划过去。
    An exception, not a warning: warnings get scrolled past."""

def ask(model, system, user, label='', temperature=0.0, max_tokens=1200):
    """HARNESS：调一次模型，记账，守预算。/ One call, ledgered and guarded."""
    if spent_cny() >= BUDGET_CNY:
        raise BudgetExceeded(f'已花 CNY {spent_cny():.4f}，达上限 {BUDGET_CNY}。'
                             f'确认要继续就调大 BUDGET_CNY 再重跑。')

    t0 = time.time()
    r = client.chat.completions.create(
        model=model,
        # ⚠️ context engineering 的全部：system 放【不变】的，user 放【每次变】的。
        #    顺序反过来 → 缓存一次都命中不了 → 贵 50 倍。
        # ⚠️ This one line IS the context engineering: stable in system,
        #    variable in user. Reversed, nothing caches and it costs 50x.
        messages=[{'role': 'system', 'content': system},
                  {'role': 'user',   'content': user}],
        temperature=temperature,   # 0 = 尽量每次一样 / deterministic-ish
        max_tokens=max_tokens,
    )
    dt, u = time.time() - t0, r.usage

    hit  = getattr(u, 'prompt_cache_hit_tokens', 0) or 0
    miss = getattr(u, 'prompt_cache_miss_tokens', None)
    if miss is None: miss = u.prompt_tokens - hit
    # 思考 token 也算输出，也要付钱 / reasoning tokens are billed as output
    think = getattr(u.completion_tokens_details, 'reasoning_tokens', 0) or 0

    p = PRICES[model]
    cost = (hit*p['hit'] + miss*p['miss'] + u.completion_tokens*p['out']) / 1_000_000
    LEDGER.append(dict(label=label, model=model, hit=hit, miss=miss,
                       out=u.completion_tokens, think=think,
                       total_in=u.prompt_tokens, cost_usd=cost, secs=dt))

    pct = 100*hit/u.prompt_tokens if u.prompt_tokens else 0
    print(f"[{label or model}] {dt:4.1f}s")
    print(f"   input  {u.prompt_tokens:>6}　cache hit {hit:>6} ({pct:5.1f}%)　miss {miss:>6}")
    print(f"   output {u.completion_tokens:>6}　其中思考 reasoning {think}")
    print(f"   这次 CNY {cost*USD_TO_CNY:.5f}　累计 {spent_cny():.4f}"
          f"　剩余额度 {BUDGET_CNY - spent_cny():.4f}")
    return r.choices[0].message.content

def estimate(n, rubric_tok, jd_tok, out_tok=400, model=None):
    """跑之前先估价。第一次 miss，之后 rubric 都 hit。"""
    p = PRICES[model or FLASH]
    miss = rubric_tok + n*jd_tok
    hit  = (n-1) * rubric_tok
    usd  = (miss*p['miss'] + hit*p['hit'] + n*out_tok*p['out']) / 1_000_000
    print(f'预估 {n} 个岗位　CNY {usd*USD_TO_CNY:.4f}（每个 {usd*USD_TO_CNY/n:.5f}）')
    return usd*USD_TO_CNY

print(f'harness 就绪　预算上限 CNY {BUDGET_CNY:.2f}')

### 🔨 刻意练习 · deliberate practice

1. 把 `ask()` 里的 `messages` 顺序对调（system 放岗位、user 放 rubric），跑两次，看 cache hit 变成多少。**看完改回来。**
2. 把 `BUDGET_CNY` 改成 `0.0001`，跑第 9 格，确认它真的会抛异常而不是继续跑。
3. 回答：为什么预算闸要抛异常而不是打印警告？

---
# Part 4 · 第一次调用 —— 和一个会耗掉你十分钟的坑

## ⚠️ `max_tokens` 太小 → 答案是空的，而且**不报错**

`deepseek-v4-flash` 会**先思考再回答**（reasoning model）。
思考也算 token，**并且从 `max_tokens` 里扣**。我实测：

```
max_tokens=20   → 答案 ''       输出 20（思考 20）  ← 全花在思考上，一个字没剩
max_tokens=200  → 答案 '惠灵顿'  输出 29（思考 25）  ← 够了
```

**空答案不会报错。** 你会以为提示词写错了、JSON 解析坏了、模型抽风。

> **教学要点：AI 的失败经常长得不像失败。**
> 这和 FlowGT 主项目里那条贯穿一切的教训是同一个形状 ——
> **一次失败被渲染成了一个空结果。**

> **这一格干什么** 故意用两个 `max_tokens` 各调一次，对比。  
> **输入 input** 无  
> **花费 cost** **花钱**，极少，约 CNY 0.0001  
> **你应该看到 expected** 两行。第一行答案是 `''`，第二行是 `'惠灵顿'`
  
> **看到别的说明** 两行都有答案 = 模型行为变了，更好，但记住这个坑

In [ ]:
for mt in (20, 200):
    r = client.chat.completions.create(model=FLASH,
        messages=[{'role':'system','content':'你是一个只回答一个词的助手。'},
                  {'role':'user','content':'用一个词回答：新西兰的首都是哪里？'}],
        temperature=0, max_tokens=mt)
    d = r.usage.completion_tokens_details
    print(f'max_tokens={mt:<4} → 答案 {r.choices[0].message.content.strip()!r:<10} '
          f'输出 {r.usage.completion_tokens}（思考 {d.reasoning_tokens}）')

---
# Part 5 · 输入从哪来 —— 以及一个坏消息

## 5.1 上下文管理：从哪开始，到哪结束

```
  ┌─ 上下文管理【开始】
  │
  │  ① 挑数据      从 155 个岗位里挑 12 个        ← 这一步就在做 context 管理
  │  ② 抓正文      每个岗位的 JD                   ← 决定「什么进窗口」
  │  ③ 截断        jd[:6000]                       ← 决定「进多少」
  │  ④ 组装        system=rubric  user=岗位        ← 决定「什么顺序」
  │  ⑤ 调用        ask(...)
  │
  └─ 上下文管理【结束】 ← 模型开始生成的那一刻

     ⑥ 解析 JSON   ← 这之后是普通程序，不再是 context 的事
     ⑦ 排序、筛选
```

**记住这条线：** context engineering 在「模型开始生成」那一刻**结束**。
之后的一切都是普通编程。很多人把两边搅在一起，于是既调不好提示词，也写不干净代码。

## 5.2 ⚠️ 坏消息：我们没有 JD 正文

Radar 的**全部前提**是读 JD 正文。但 `flowgt-job-hunter/jobs.db` 里 155 个岗位，
**JD 字段全是空的**。爬虫在 `scraper.py` 第 291 行定义了 `fetch_jd()`，
**整个仓库没有一处调用它** —— 写了，忘了接上。

下面先**证明**这件事，而不是听我说。

> **这一格干什么** 数一下有多少岗位、多少有正文。  
> **输入 input** `../flowgt-job-hunter/jobs.db`  
> **花费 cost** 不花钱  
> **你应该看到 expected** `岗位总数 155　有 JD 正文的 0`
  
> **看到别的说明** 如果不是 0，说明爬虫已经接上了，抓正文那格可以跳过

In [ ]:
import sqlite3
DB = pathlib.Path('../flowgt-job-hunter/jobs.db')
assert DB.exists(), f'找不到 {DB} —— 确认两个仓库同级'
con = sqlite3.connect(DB)
n_all, = con.execute('SELECT COUNT(*) FROM jobs').fetchone()
n_jd,  = con.execute('SELECT COUNT(*) FROM jobs WHERE jd IS NOT NULL AND length(jd)>200').fetchone()
print(f'岗位总数 {n_all}　有 JD 正文的 {n_jd}')
print('→ 没有输入，下面现场抓。' if n_jd == 0 else '→ 有正文，可直接用。')

> **这一格干什么** 挑最近的 12 个岗位。  
> **输入 input** `jobs.db`  
> **花费 cost** 不花钱  
> **你应该看到 expected** `准备抓 12 个…` + 前 5 个标题
  
> **看到别的说明** 一个都没有 = 库是空的，先去 flowgt-job-hunter 跑 `bin/hunt-local`

In [ ]:
rows = con.execute('''
    SELECT id, job_title, company, location, job_url
    FROM jobs WHERE job_url IS NOT NULL
    ORDER BY scraped_at DESC LIMIT ?''', (MAX_JOBS,)).fetchall()
print(f'准备抓 {len(rows)} 个岗位的正文…')
for _, t, c, _, _ in rows[:5]: print('  ', t[:50], '@', c)

> **这一格干什么** 用 Playwright 逐个打开岗位页取正文。  
> **输入 input** 上一格的 12 个链接  
> **花费 cost** **不花 API 的钱**，只花时间（约 1 分钟）  
> **你应该看到 expected** 12 行进度 + `拿到 N 份`，以及平均长度（这个数决定每个岗位多少钱）
  
> **看到别的说明** 全失败 = Seek 改版了。跑 `flowgt-job-hunter/audit_seek.py` 看现在页面长什么样

In [ ]:
import asyncio
from playwright.async_api import async_playwright

async def grab_jds(rows):
    jobs = []
    async with async_playwright() as pw:
        b = await pw.chromium.launch(headless=True,
            args=['--disable-blink-features=AutomationControlled'])
        ctx = await b.new_context(viewport={'width':1280,'height':800},
            user_agent='Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
                       '(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36')
        await ctx.add_init_script(
            "Object.defineProperty(navigator,'webdriver',{get:()=>undefined});")
        page = await ctx.new_page()
        for i, (jid, title, co, loc, url) in enumerate(rows, 1):
            try:
                await page.goto(url, wait_until='domcontentloaded', timeout=25000)
                await asyncio.sleep(1.2)   # 礼貌延迟 / be polite to the site
                el = None
                # 三个选择器轮着试：Seek 改过版，多一条退路多一分活路
                for sel in ['[data-automation="jobAdDetails"]',
                            '[data-automation="jobDescription"]',
                            '[data-testid="job-detail-overview"]']:
                    el = await page.query_selector(sel)
                    if el: break
                jd = (await el.inner_text()) if el else ''
                jobs.append(dict(id=jid, title=title, company=co,
                                 location=loc, url=url, jd=jd))
                print(f'  {i:>2}/{len(rows)}  {len(jd):>5} 字  {title[:44]}')
            except Exception as e:
                print(f'  {i:>2}/{len(rows)}  失败 {type(e).__name__}  {title[:40]}')
        await b.close()
    return jobs

JOBS = await grab_jds(rows)
JOBS = [j for j in JOBS if len(j['jd']) > 300]
avg = sum(len(j['jd']) for j in JOBS) / max(1, len(JOBS))
print(f'\n拿到 {len(JOBS)} 份有正文的岗位')
print(f'平均 {avg:.0f} 字符 ≈ {avg/3.5:.0f} tokens —— 这决定每个岗位多少钱')

---
# Part 6 · rubric —— 它就是一个 Skill

这一段**每次调用都一模一样**，所以放 `system`、放最前面 → 被缓存。

它同时是 Part 1.4 讲的那个 **Agent Skill**：
一门专门知识，和调用代码**分开存放**，可以被单独修改、单独版本管理。

> **它是 v0.1，本来就该被你改。**
> 跑完看结果，哪条判断和你的直觉不一样，就回来改哪条 ——
> **改完它才是「你的」雷达**，而不是我的。

> **这一格干什么** 定义 rubric 字符串。  
> **输入 input** 无  
> **花费 cost** 不花钱  
> **你应该看到 expected** 长度（字符 + token 估算）

In [ ]:
RUBRIC = '''你是新西兰 IT 就业市场的分析员。给你一份职位广告，你只输出 JSON。

给两个 0-5 的分：

【AI 渗透度 ai_depth】这份工作实际会碰到多少 AI
  5 = 标题就是 AI/ML/Agent 工程师，整份 JD 围绕模型
  4 = 标题是普通开发，但职责前三条有一条是 AI
  3 = AI 出现在加分项或未来路线图里　←【甜蜜区 sweet spot】
  2 = 只有机器学习/数据底子，没有明确 AI
  1 = 只有公司简介提了一句 AI-powered
  0 = 完全没有

  显性信号 explicit：LLM GPT Claude RAG agent prompt engineering embedding
      vector database fine-tuning LangChain LlamaIndex MCP function calling
  隐性信号 implicit：machine learning MLOps NLP computer vision
      recommendation forecasting anomaly detection data platform feature store
  前瞻信号 forward-looking：AI-first、exploring AI/ML、automation roadmap、
      intelligent features、we are just starting to、you will help shape、greenfield

【新人友好度 newcomer_fit】一个转行的人投了有没有戏
  加分：0-2 年 / graduate / early career；没要求新西兰本地经验；
        愿意 sponsor 或只要求 eligible to work；在扩张、招多个；
        JD 写得笼统留有余地；技术栈主流可自学（Python / TypeScript / cloud）
  减分：要 5 年以上、proven track record、lead；
        硬性要求 NZ experience【最大的墙】；必须公民或永居；
        只招一个且是替补某个走掉的 senior；列了十几条硬性技术要求；
        冷门专有栈或需要行业年限（医疗器械、航电等）

只输出这个 JSON，不要解释文字、不要 markdown 代码块：
{"ai_depth": 0-5, "newcomer_fit": 0-5,
 "ai_signals": ["正文里命中的原词"],
 "barriers": ["抬高门槛的原句"],
 "one_line": "一句中文说清这岗位对新人意味着什么"}'''

print(f'rubric {len(RUBRIC)} 字符 ≈ {len(RUBRIC)/1.7:.0f} tokens')
print('这段每次都相同 → 被缓存 → 第二次起便宜约 50 倍')

---
# Part 7 · **先估价，再花钱**

> 「我以为很便宜」和「我算过是多少」之间，隔着一次超支。

> **这一格干什么** 估算 12 个岗位的花费。  
> **输入 input** rubric 长度 + JD 平均长度  
> **花费 cost** 不花钱  
> **你应该看到 expected** 一行 CNY 金额。按目前量级应该是**几分钱**
  
> **看到别的说明** 如果超过 `BUDGET_CNY`，把 `MAX_JOBS` 调小

In [ ]:
rubric_tok = int(len(RUBRIC)/1.7)
jd_tok     = int(avg/3.5)
est = estimate(len(JOBS), rubric_tok, jd_tok, out_tok=400, model=FLASH)
print(f'\n预算 CNY {BUDGET_CNY:.2f}　预估 {est:.4f}　'
      f"→ {'放心跑' if est < BUDGET_CNY*0.5 else '⚠️ 接近上限，减少岗位数'}")

---
# Part 8 · 跑起来

## 8.1 第一个岗位 —— 看 **cache miss** 长什么样

> **这一格干什么** 定义 `score_job()`，评第一个岗位。  
> **输入 input** `JOBS[0]` + `RUBRIC`  
> **花费 cost** **花钱**，约 CNY 0.005  
> **你应该看到 expected** 台账几行 —— 注意 **cache hit 0 (0.0%)**，第一次谁也命中不了。然后是 JSON
  
> **看到别的说明** `⚠️ 没解析出 JSON` = 多半是 `max_tokens` 不够（Part 4 那个坑）

In [ ]:
def score_job(job, model=FLASH, label=None):
    """评一个岗位。rubric→system（缓存），岗位→user（每次变）。"""
    user = (f"职位：{job['title']}\n公司：{job['company']}\n地点：{job['location']}\n\n"
            # 截断到 6000 字符：context window 有限，而且每个字都要付钱。
            # 这一行就是 Part 5.1 说的「决定进多少」。
            f"广告正文：\n{job['jd'][:6000]}")
    raw = ask(model, RUBRIC, user, label=label or job['title'][:26], max_tokens=1200)
    txt = raw.strip().removeprefix('```json').removeprefix('```').removesuffix('```').strip()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        # 模型没按格式来。harness 的职责之一就是【不让这个把程序打崩】。
        print('   ⚠️ 没解析出 JSON：', txt[:160])
        return None

first = score_job(JOBS[0])
print()
print(json.dumps(first, ensure_ascii=False, indent=2))

## 8.2 第二个 —— 看 **cache hit**

同样的 rubric，不同的岗位。**命中率应该跳到 60–95%。**

> 👉 **把这一格的「这次 CNY」和上一格对比。那个差价就是 prompt caching 的价值。**

> **这一格干什么** 评第二个岗位。  
> **输入 input** `JOBS[1]`  
> **花费 cost** **花钱**，应该比上一格便宜很多  
> **你应该看到 expected** cache hit 大幅上升，「这次 CNY」明显低于上一格
  
> **看到别的说明** 命中还是 0 = rubric 太短（<1000 tok），或两次间隔太久缓存过期

In [ ]:
second = score_job(JOBS[1])
print()
print(json.dumps(second, ensure_ascii=False, indent=2))

## 8.3 批量

⚠️ 连着调 10 次左右。预算闸还在守着。

> **这一格干什么** 把剩下的都评一遍，算机会指数。  
> **输入 input** `JOBS` 全部  
> **花费 cost** **花钱**，约 CNY 0.02  
> **你应该看到 expected** 每个岗位一组台账。**看着 cache hit 一路保持高位**
  
> **看到别的说明** 抛 `BudgetExceeded` = 到上限了

In [ ]:
RESULTS = []
for j in JOBS:
    try:
        s = score_job(j)
    except BudgetExceeded as e:
        print('⛔', e); break
    if s:
        s['title'], s['company'], s['url'] = j['title'], j['company'], j['url']
        # ★ 乘法，不是加法。任何一个是 0，这岗位对会员就毫无意义。
        # ★ Multiply, never add: either factor at zero makes the role useless.
        s['score'] = s['ai_depth'] * s['newcomer_fit']
        RESULTS.append(s)
    print()
print(f'评完 {len(RESULTS)} 个')

> **这一格干什么** 排序、打印表格。  
> **输入 input** `RESULTS`  
> **花费 cost** 不花钱  
> **你应该看到 expected** 一张表，分数从高到低。≥9 分的多一行点评
  
> **看到别的说明** 全是 0 分 = rubric 和这批岗位对不上，回 Part 6 改

In [ ]:
RESULTS.sort(key=lambda r: -r['score'])
print(f"{'指数':>4} {'AI':>3} {'新人':>4}  {'岗位':<40} {'公司':<20}")
print('─'*88)
for r in RESULTS:
    print(f"{r['score']:>4} {r['ai_depth']:>3} {r['newcomer_fit']:>4}  "
          f"{r['title'][:38]:<40} {(r['company'] or '')[:18]:<20}")
    if r['score'] >= 9:
        print(f"       ↳ {r['one_line']}")

---
# Part 9 · 产出：同一批数据，两边读

## 9.1 朝会员这一头 —— 捡漏名单

**标题里没有 AI 字样、但正文里有。** 这些就是关键词搜不到的岗位 ——
**Radar 存在的全部理由。**

> **这一格干什么** 筛出「标题不提 AI、渗透度 ≥3」的岗位。  
> **输入 input** `RESULTS`  
> **花费 cost** 不花钱  
> **你应该看到 expected** 若干岗位，每个带正文里的 AI 痕迹和链接
  
> **看到别的说明** 一个都没有 = 这 12 个样本里恰好没有。换一批，或把 `>=3` 放宽到 `>=2`

In [ ]:
AI_WORDS = ('ai', 'machine learning', 'ml ', 'data scien', 'llm', 'artificial')
hidden = [r for r in RESULTS if r['ai_depth'] >= 3
          and not any(w in r['title'].lower() for w in AI_WORDS)]

print(f'{len(hidden)} 个「捡漏」岗位 —— 标题不提 AI，正文在做 AI：\n')
for r in hidden:
    print(f"  {r['title']} @ {r['company']}")
    print(f"     AI {r['ai_depth']}/5 · 新人友好 {r['newcomer_fit']}/5 · 指数 {r['score']}")
    print(f"     正文痕迹：{', '.join(r['ai_signals'][:5])}")
    print(f"     {r['url']}\n")

## 9.2 朝雇主这一头 —— 「三杯咖啡」的开场白

新人友好度**低** = 门槛高 = 招不到人。**同一个分，反过来读。**

⚠️ **12 个样本说不出「市场最苛刻的 15%」这种话。**
样本小的时候，诚实的做法是**把样本量说出来** ——
下面的代码打印的是「最近 N 个里排前 M」，不是「市场前 15%」。

> **这是一条职业操守，不是代码细节。** 用一个撑不住的数字去开场，
> 对方一旦追问就全盘皆输。

> **这一格干什么** 找出门槛最高的几个，生成开场白模板。  
> **输入 input** `RESULTS`  
> **花费 cost** 不花钱  
> **你应该看到 expected** 门槛最高的岗位 + 具体门槛句子 + 一段可直接用的开场白

In [ ]:
harsh = sorted(RESULTS, key=lambda r: r['newcomer_fit'])
cut = max(1, round(len(harsh) * 0.15))
print(f'样本 {len(RESULTS)} 个 —— 门槛最高的 {cut} 个：\n')
for r in harsh[:cut]:
    print(f"  {r['title']} @ {r['company']}　新人友好度 {r['newcomer_fit']}/5")
    for b in r['barriers'][:4]: print(f'     · {b}')
    print()
print('可以这样开场（数字是真的，样本量也说出来）：')
print(f'  「我每天读新西兰全部 IT 岗位。最近这 {len(RESULTS)} 个里，')
print(f'   你们这个岗位的门槛排在最高的 {cut} 个之内 ——')
print('   要 X 年经验 + 本地经验 + N 条硬性要求。这大概率就是投的人少的原因。」')

---
# Part 10 · 结账 —— 和官方对一次账

> **这一格干什么** 汇总台账，跟官方余额核对。  
> **输入 input** `LEDGER` + 一次免费查询  
> **花费 cost** 不花钱  
> **你应该看到 expected** 总花费、cache 命中率、每岗位均价，以及「我算的」vs「官方扣的」
  
> **看到别的说明** 两个数字差很多 = 价目表过期了，回 Part 3 核价

In [ ]:
usd  = sum(r['cost_usd'] for r in LEDGER)
hit  = sum(r['hit'] for r in LEDGER)
miss = sum(r['miss'] for r in LEDGER)
out  = sum(r['out'] for r in LEDGER)

print(f'调用次数     {len(LEDGER)}')
print(f'input tokens {hit+miss:,}　cache hit {hit:,} ({100*hit/max(1,hit+miss):.1f}%)')
print(f'output tokens{out:,}')
print(f'总花费       CNY {usd*USD_TO_CNY:.4f}')
print(f'每岗位均价   CNY {usd*USD_TO_CNY/max(1,len(RESULTS)):.5f}')

# ★ 和官方对账。自己算的和人家扣的对不上 = 价目表该更新了。
live = api('/user/balance')
now = float(next(b for b in live['balance_infos'] if b['currency']=='CNY')['total_balance'])
print(f'\n开始余额 CNY {START_BALANCE:.2f}')
print(f'现在余额 CNY {now:.2f}　← 官方数字')
print(f'官方扣了 CNY {START_BALANCE-now:.4f}　我算的 CNY {usd*USD_TO_CNY:.4f}')

> **这一格干什么** 把单价推算到真实规模。  
> **输入 input** 上一格的每岗位均价  
> **花费 cost** 不花钱  
> **你应该看到 expected** 「每天 80 个 × 22 个工作日 = 每月 CNY X ≈ NZ$ Y」和 NZ$15 预算线的对比

In [ ]:
per_job = usd * USD_TO_CNY / max(1, len(RESULTS))
m = per_job * 80 * 22
print(f'每岗位 CNY {per_job:.5f}')
print(f'每天 80 个 × 22 工作日 = 每月 CNY {m:.2f} ≈ NZ$ {m/4.2:.2f}')
print()
print('对照 CLAUDE.md：「API 成本 <NZ$15/月，不构成任何约束」')
print('→ 远低于 15 的话，当初以成本为由把这功能搁置的理由就不成立了。')

---
# Part 11 · 刻意练习 · deliberate practice

**跑通不等于学会。** 下面每一条都要动手，不要只是读。

## A · 摸清成本（改参数，看数字变化）

1. 把 `ask()` 里 `messages` 的 system 和 user 对调，跑两次，记下 cache hit。**再改回来。**
2. 把 `score_job` 的 `jd[:6000]` 改成 `jd[:1500]`，重跑，对比：省了多少钱？评分变差了吗？
3. 把 `FLASH` 换成 `PRO` 评同一个岗位。贵了几倍？判断真的更好吗？
4. 把 `BUDGET_CNY` 设成 `0.0001`，确认预算闸真的会拦住。

## B · 摸清判断（改 rubric，看结果变化）

5. 从 Part 8 的表里挑 **3 个你不同意的评分**。写下你认为应该是几分、为什么。
6. 回 Part 6 改 rubric，把你的判断写进去。重跑。**有没有变成你要的样子？**
7. 如果没变：是 rubric 写得不够具体，还是模型没读懂？怎么区分这两种？

## C · 摸清边界（想清楚，不写代码）

8. Radar 是 workflow 还是 agent？如果要把它变成 agent，得加什么？**值得吗？**
9. 如果 Seek 明天改版，这个笔记本哪一格会先坏？你怎么知道它坏了？
10. 12 个样本能说「市场前 15%」吗？要多少个才能说？**说不了的时候该怎么说话？**

## D · 费曼验收 · teach it back

11. 不看笔记本，在白板上画出 Part 0 的图，讲给一个组员听。
12. 用一句话解释 **prompt caching**，再用一个数字支撑它。
13. 解释 **harness** 是什么，说出三件没有它就做不到的事。
14. 解释 **MCP** 和 **Skill** 的区别，各用一个比喻。

> **讲不出来就是没学会。** 这是这份笔记本的验收标准。

---
## 跑完之后，真正的下一步

1. **改 rubric。** 练习 5–7。改完它才是**你的**雷达。
2. **把 `fetch_jd()` 接上。** 爬虫定义了却从没调用，所以库里没有正文。
   接上之后 Radar 每天有 80 个岗位可读，而不是这里临时抓的 12 个。
3. **别急着上线。** 先回答：**这周的雇主对话数是几？**
   Radar 是给雇主对话用的弹药 —— 没有对话，就没有开枪的地方。

### 这个笔记本【故意】没做的事

| 没做 | 为什么 |
|---|---|
| 两段式（flash 粗筛 → pro 细读） | 12 个岗位用不上。真跑 80 个时再加，能再省一半 |
| 存结果到数据库 | 先证明有没有用，再考虑落库 |
| 离线评测（拿历史点击当答案） | 那是另一个笔记本，也是决定「值不值得上线」的那一个 |
| 绕过 Cloudflare | 那是反机器人检测，不碰 |
| MCP | 这里模型不需要自己去查任何东西。**用不上的标准不要引入** |

### 📖 参考文档 · references

- Anthropic《Building Effective Agents》—— workflow vs agent 的官方定义　
  <https://www.anthropic.com/engineering/building-effective-agents>
- Anthropic《Agent Skills》—— skill 结构与 progressive disclosure　
  <https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills>
- Model Context Protocol —— <https://modelcontextprotocol.io>
- DeepSeek API 文档（价格、缓存字段）—— <https://api-docs.deepseek.com>